# 02 — Preprocessing Dataset DapurFinder

## Tujuan
Tahap preprocessing dilakukan untuk menyiapkan dataset resep sebelum digunakan dalam sistem pencarian resep DapurFinder.

Proses yang dilakukan meliputi:
1. Membaca dataset resep hasil penggabungan.
2. Memeriksa dan menghapus data yang tidak memiliki judul, bahan, atau langkah memasak.
3. Menghapus resep duplikat berdasarkan judul, bahan, dan langkah memasak yang telah dinormalisasi.
4. Memeriksa kembali kualitas dataset setelah dibersihkan.
5. Menyimpan dataset hasil preprocessing untuk digunakan pada tahap implementasi sistem pencarian.

Dataset asli tidak diubah. Seluruh hasil preprocessing disimpan sebagai file baru di folder `dataset` yang tidak diunggah ke GitHub.

In [1]:

import pandas as pd
from pathlib import Path

# Menentukan folder utama proyek secara otomatis
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Lokasi dataset
DATA_PATH = ROOT_DIR / "dataset" / "recipes.csv"

# Membaca dataset asli
df = pd.read_csv(DATA_PATH)

print("Lokasi proyek:", ROOT_DIR)
print("Ukuran dataset asli:", df.shape)
print("Nama kolom:", df.columns.tolist())

df.head()


Lokasi proyek: D:\STKI-Recipe-Finder
Ukuran dataset asli: (15641, 6)
Nama kolom: ['Title', 'Ingredients', 'Steps', 'Loves', 'URL', 'Category']


,Title,Ingredients,Steps,Loves,URL,Category
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,Cuci bersih ayam dan tiriskan. Lalu peras jeru...,1,/id/resep/4473027-ayam-woku-manado,ayam
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"Haluskan bumbu2nya (BaPut, ketumbar, kemiri, k...",1,/id/resep/4471956-ayam-goreng-tulang-lunak,ayam
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,Panaskan minyak di dalam wajan. Setelah minyak...,2,/id/resep/4473057-ayam-cabai-kawin,ayam
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,Goreng ayam seperti ayam krispi--Ulek semua ba...,10,/id/resep/4473023-ayam-geprek,ayam
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,Cuci bersih kulit ayam. Sisihkan--Ambil 50 ml ...,4,/id/resep/4427438-minyak-ayam,ayam


In [2]:

kolom_wajib = ["Title", "Ingredients", "Steps"]

# Memastikan kolom yang dibutuhkan tersedia
kolom_tidak_ada = [kolom for kolom in kolom_wajib if kolom not in df.columns]

if kolom_tidak_ada:
    raise ValueError(f"Kolom wajib tidak ditemukan: {kolom_tidak_ada}")

print("Semua kolom wajib tersedia.")
print("\nJumlah nilai kosong sebelum preprocessing:")
print(df[kolom_wajib].isna().sum())


Semua kolom wajib tersedia.

Jumlah nilai kosong sebelum preprocessing:
Title           0
Ingredients    48
Steps          46
dtype: int64


In [3]:

# Membuat salinan agar dataset asli tidak berubah
df_clean = df.copy()

# Menganggap teks kosong atau hanya berisi spasi sebagai nilai kosong
for kolom in kolom_wajib:
    df_clean[kolom] = df_clean[kolom].replace(
        r"^\s*$", pd.NA, regex=True
    )

jumlah_awal = len(df_clean)

# Menghapus baris tanpa judul, bahan, atau langkah memasak
df_clean = df_clean.dropna(subset=kolom_wajib).copy()

jumlah_setelah_hapus_kosong = len(df_clean)
jumlah_dihapus_kosong = jumlah_awal - jumlah_setelah_hapus_kosong

print("Jumlah data awal:", jumlah_awal)
print("Data dihapus karena kolom wajib kosong:", jumlah_dihapus_kosong)
print("Jumlah data setelah menghapus nilai kosong:", jumlah_setelah_hapus_kosong)


Jumlah data awal: 15641
Data dihapus karena kolom wajib kosong: 48
Jumlah data setelah menghapus nilai kosong: 15593


In [4]:

def normalisasi_teks(teks):
    """Merapikan spasi dan menyamakan huruf untuk perbandingan teks."""
    return " ".join(str(teks).casefold().split())

# Membuat kunci pembanding tanpa mengubah kolom asli
df_clean["_key_title"] = df_clean["Title"].map(normalisasi_teks)
df_clean["_key_ingredients"] = df_clean["Ingredients"].map(normalisasi_teks)
df_clean["_key_steps"] = df_clean["Steps"].map(normalisasi_teks)

jumlah_sebelum_duplikat = len(df_clean)

# Menghapus resep dengan judul, bahan, dan langkah yang sama
df_clean = df_clean.drop_duplicates(
    subset=["_key_title", "_key_ingredients", "_key_steps"],
    keep="first"
).copy()

jumlah_duplikat_dihapus = jumlah_sebelum_duplikat - len(df_clean)

# Menghapus kolom bantu agar tidak ikut menjadi kolom dataset akhir
df_clean = df_clean.drop(
    columns=["_key_title", "_key_ingredients", "_key_steps"]
)

print("Data sebelum penghapusan duplikat:", jumlah_sebelum_duplikat)
print("Duplikat yang dihapus:", jumlah_duplikat_dihapus)
print("Data setelah penghapusan duplikat:", len(df_clean))


Data sebelum penghapusan duplikat: 15593
Duplikat yang dihapus: 650
Data setelah penghapusan duplikat: 14943


In [5]:

print("=== HASIL VALIDASI DATASET ===")
print("Jumlah baris:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))

print("\nNilai kosong pada kolom wajib:")
print(df_clean[kolom_wajib].isna().sum())

print("\nJumlah duplikat seluruh kolom:", df_clean.duplicated().sum())

print("\nJumlah resep per kategori:")
print(df_clean["Category"].value_counts())

df_clean.head()


=== HASIL VALIDASI DATASET ===
Jumlah baris: 14943
Jumlah kolom: 6

Nilai kosong pada kolom wajib:
Title          0
Ingredients    0
Steps          0
dtype: int64

Jumlah duplikat seluruh kolom: 0

Jumlah resep per kategori:
Category
sapi       1933
ikan       1932
ayam       1901
tahu       1892
kambing    1877
udang      1870
telur      1790
tempe      1748
Name: count, dtype: int64


,Title,Ingredients,Steps,Loves,URL,Category
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,Cuci bersih ayam dan tiriskan. Lalu peras jeru...,1,/id/resep/4473027-ayam-woku-manado,ayam
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"Haluskan bumbu2nya (BaPut, ketumbar, kemiri, k...",1,/id/resep/4471956-ayam-goreng-tulang-lunak,ayam
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,Panaskan minyak di dalam wajan. Setelah minyak...,2,/id/resep/4473057-ayam-cabai-kawin,ayam
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,Goreng ayam seperti ayam krispi--Ulek semua ba...,10,/id/resep/4473023-ayam-geprek,ayam
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,Cuci bersih kulit ayam. Sisihkan--Ambil 50 ml ...,4,/id/resep/4427438-minyak-ayam,ayam


In [6]:

OUTPUT_PATH = ROOT_DIR / "dataset" / "recipes_clean.csv"

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Dataset bersih berhasil disimpan.")
print("Lokasi:", OUTPUT_PATH)
print("Jumlah baris:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))
print("Ukuran file (KB):", round(OUTPUT_PATH.stat().st_size / 1024, 2))


Dataset bersih berhasil disimpan.
Lokasi: D:\STKI-Recipe-Finder\dataset\recipes_clean.csv
Jumlah baris: 14943
Jumlah kolom: 6
Ukuran file (KB): 12055.65


## Kesimpulan

Tahap preprocessing berhasil dilakukan pada dataset DapurFinder. Dari 15.641 baris data awal, sebanyak 48 baris dihapus karena memiliki nilai kosong pada kolom wajib, kemudian 650 baris duplikat dihapus berdasarkan judul, bahan, dan langkah memasak yang telah dinormalisasi.

Hasil akhir preprocessing berjumlah 14.943 resep dengan enam kolom. Pemeriksaan menunjukkan bahwa kolom Title, Ingredients, dan Steps tidak memiliki nilai kosong, serta tidak ditemukan duplikat berdasarkan seluruh kolom.

Dataset bersih disimpan secara terpisah sebagai `recipes_clean.csv` sehingga data asli tetap terjaga. Dataset ini selanjutnya akan digunakan untuk membangun sistem pencarian resep menggunakan metode TF-IDF dan Cosine Similarity.